In [1]:
# ================================
# 📌 1. IMPORTS
# ================================
import pandas as pd
from collections import defaultdict
import random

# ================================
# 📌 2. LOAD DATA
# ================================
ratings = pd.read_csv("/content/ratings.csv")
movies = pd.read_csv("/content/movies_metadata.csv", low_memory=False)

# Fix movie ID type
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')

# ================================
# 📌 3. PREPROCESSING
# ================================
ratings = ratings.sort_values(by=["userId", "timestamp"])

# Reduce dataset size (IMPORTANT 🚨)
top_movies = ratings["movieId"].value_counts().head(500).index
ratings = ratings[ratings["movieId"].isin(top_movies)]

# ================================
# 📌 4. BUILD TRANSITIONS
# ================================
def build_transitions(ratings_df):
    transitions = []

    for user_id, user_ratings in ratings_df.groupby('userId'):
        movie_seq = user_ratings["movieId"].tolist()

        for i in range(len(movie_seq) - 1):
            state = movie_seq[i]
            next_state = movie_seq[i + 1]

            transitions.append((state, next_state))

    return transitions

transitions = build_transitions(ratings)

# ================================
# 📌 5. BUILD PROBABILITY TABLE (Bayesian Model)
# ================================
class BayesianRecommender:
    def __init__(self, transitions):
        self.transition_counts = defaultdict(lambda: defaultdict(int))
        self.transition_probs = defaultdict(dict)
        self.build_model(transitions)

    def build_model(self, transitions):
        # Count transitions
        for state, next_state in transitions:
            self.transition_counts[state][next_state] += 1

        # Convert counts → probabilities
        for state in self.transition_counts:
            total = sum(self.transition_counts[state].values())

            for next_state in self.transition_counts[state]:
                self.transition_probs[state][next_state] = (
                    self.transition_counts[state][next_state] / total
                )

    def recommend(self, state, top_n=5):
        if state not in self.transition_probs:
            return []

        sorted_movies = sorted(
            self.transition_probs[state].items(),
            key=lambda x: x[1],
            reverse=True
        )

        return [movie for movie, _ in sorted_movies[:top_n]]

# ================================
# 📌 6. TRAIN MODEL
# ================================
bayes_model = BayesianRecommender(transitions)

# ================================
# 📌 7. MOVIE ID → TITLE
# ================================
movie_titles = movies[['id', 'title']].dropna()
movie_dict = dict(zip(movie_titles['id'], movie_titles['title']))

# ================================
# 📌 8. TEST RECOMMENDATION
# ================================
test_movie = random.choice(list(ratings["movieId"].unique()))

recommendations = bayes_model.recommend(test_movie)

print("🎬 Current Movie:", movie_dict.get(test_movie, test_movie))
print("\n🔥 Recommended Movies:\n")

for movie_id in recommendations:
    print(movie_dict.get(movie_id, f"Movie ID: {movie_id}"))

🎬 Current Movie: We Own the Night

🔥 Recommended Movies:

Bell, Book and Candle
Movie ID: 2081.0
Pleasantville
Movie ID: 5952.0
Anatomy of Hell


In [1]:
import pandas as pd
from collections import defaultdict
import random

In [2]:
ratings = pd.read_csv("C:/Users/devya/OneDrive/Desktop/AML/exp_4/archive (4)/ratings.csv", low_memory=False)
movies = pd.read_csv("C:/Users/devya/OneDrive/Desktop/AML/exp_4/archive (4)/movies_metadata.csv", low_memory=False)

In [3]:
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')

In [4]:
ratings = ratings.sort_values(by=["userId", "timestamp"])

# Reduce dataset size (IMPORTANT 🚨)
top_movies = ratings["movieId"].value_counts().head(500).index
ratings = ratings[ratings["movieId"].isin(top_movies)]

In [5]:
def build_transitions(ratings_df):
    transitions = []

    for user_id, user_ratings in ratings_df.groupby('userId'):
        movie_seq = user_ratings["movieId"].tolist()

        for i in range(len(movie_seq) - 1):
            state = movie_seq[i]
            next_state = movie_seq[i + 1]

            transitions.append((state, next_state))

    return transitions

transitions = build_transitions(ratings)

In [6]:
class BayesianRecommender:
    def __init__(self, transitions):
        self.transition_counts = defaultdict(lambda: defaultdict(int))
        self.transition_probs = defaultdict(dict)
        self.build_model(transitions)

    def build_model(self, transitions):
        # Count transitions
        for state, next_state in transitions:
            self.transition_counts[state][next_state] += 1

        # Convert counts → probabilities
        for state in self.transition_counts:
            total = sum(self.transition_counts[state].values())

            for next_state in self.transition_counts[state]:
                self.transition_probs[state][next_state] = (
                    self.transition_counts[state][next_state] / total
                )

    def recommend(self, state, top_n=5):
        if state not in self.transition_probs:
            return []

        sorted_movies = sorted(
            self.transition_probs[state].items(),
            key=lambda x: x[1],
            reverse=True
        )

        return [movie for movie, _ in sorted_movies[:top_n]]

In [7]:
bayes_model = BayesianRecommender(transitions)

In [8]:
movie_titles = movies[['id', 'title']].dropna()
movie_dict = dict(zip(movie_titles['id'], movie_titles['title']))

In [9]:
test_movie = random.choice(list(ratings["movieId"].unique()))

recommendations = bayes_model.recommend(test_movie)

print("🎬 Current Movie:", movie_dict.get(test_movie, test_movie))
print("\n🔥 Recommended Movies:\n")

for movie_id in recommendations:
    print(movie_dict.get(movie_id, f"Movie ID: {movie_id}"))

🎬 Current Movie: 48394

🔥 Recommended Movies:

Boat
Cheerleaders' Wild Weekend
Miffo
Movie ID: 4878
Movie ID: 3949
